---
title: "Resultados potenciales"
description: "Modelo de resultados potenciales y mecanismo de asignación"
categories: [Causal Inference]
order: 2
---

In [ ]:
import numpy as np
import pandas as pd

# Resultados potenciales

## Notación de resultados potenciales

<br>

**Definición: Resultados potenciales $Y_i(1)$ y $Y_i(0)$**

Consideremos un estudio con $n$ unidades cuyo índice es $i = 1, 2,...,n$ en donde el tratamiento tiene dos niveles:
 * $1$ para el tratamiento,
 * $0$ para el control.

 Con esto, cada unidad $i$ tiene dos versiones del resultado de interés:
 $$Y_i(1), Y_i(0)$$

los cuales son llamados *resultados potenciales* bajo las intervenciones $1$ y $0$, respectivamente.


<br>

**SUPUESTOS**

La definición anterior conlleva dos

***1. Supuesto de no interferencia***: Los resultados potenciales de la unidad $i$ no dependen de los resultados potenciales de otra unidad $j$ ni viceversa, los resultados potenciales de la unidad $j$ no dependen de los resultados potenciales de la unidad $i$.

***2. Supuesto de consistencia***: No existen otras versiones del tratamiento. Lo anterior equivale a decir que el tratamiento se encuentra bien definido.

Los supuestos 1 y 2, combinados, dan el supuesto número 3: *Stable Unit Treatment Value Assumption* o conocido como *SUTVA*

***3. SUTVA***: Bajo SUTVA y un tratamiento de dos niveles, se puede formar la *tabla de ciencia*

| $i$ | $Y_i(1)$ | $Y_i(0)$ |
|-----|----------|----------|
| 1 | $Y_1(1)$ | $Y_1(0)$ |
| 2 | $Y_2(1)$ | $Y_2(0)$ |
| $\vdots$ | $\vdots$ | $\vdots$ |
| $n$ | $Y_n(1)$ | $Y_n(0)$ |

<br>

**Definición: efectos causales $\tau_i$**

Para cada unidad $i$, los efectos causales se definen como la resta de los efectos de aplicar el tratamiento y el control (notese que esto es asumiendo que es posible observar ambos resultados potenciales).

$$\tau_i = Y_I(1) - Y_I(0)$$

Sin embargo, en la realidad solo podemos observar uno de los dos resultados potenciales, es decir, la mitad de la *Tabla de Ciencia*.

<br>

**Definición: efectos causales $\tau_i$**

Para cada unidad $i$, los efectos causales se definen como la resta de los efectos de aplicar el tratamiento y el control (notese que esto es asumiendo que es posible observar ambos resultados potenciales).

$$\tau_i = Y_I(1) - Y_I(0)$$

Sin embargo, en la realidad solo podemos observar uno de los dos resultados potenciales, es decir, la mitad de la *Tabla de Ciencia*.

<br>

**Definición: Efecto Causal Promedio (ACE o ATE) $\tau$**

El Efecto Causal Promedio (ACE o ATE) es el promedio de los efectos causales.

$$\tau = n^{-1}\sum_{i=1}^{n}\{Y_i(1) - Y_i(0)\} = n^{-1}\sum_{i=1}^{n}Y_i(1) - n^{-1}\sum_{i=1}^{n}Y_i(0)$$


In [ ]:
# Efectos causales, subgrupos y la no existencia de la paradoja de Yule-Simpson

# En este ejemplo, vamos a calcular los efectos causales de subgrupos
# En este caso, necesitamos generar dependencia por grupo y generar dependencia
# del efecto potencial de aplicar el tratamiento, dado el efecto potencial del control

x = np.random.binomial(1, 0.5, 100)
y_0 = np.where(x==1, np.random.normal(8,1,100), np.random.normal(4,1,100) ) # dependencia e
y_1 = y_0 + np.random.normal(loc = 2, scale = 0.5, size = 100)
tau = y_1 - y_0
n = np.array([100 - np.sum(x), np.sum(x)])

df = pd.DataFrame({"y_1":y_1, "y_0":y_0, "x":x, "tau_grupo":tau})

df_summary = df.groupby("x").mean()
df_summary["n"] = n/np.sum(n)

# tau_grupo presenta el efecto causal promedio del subgrupo.
print("Efectos causales por grupo:\n",df_summary['tau_grupo'])

# El efecto causal promedio global es el promedio ponderado de tau_grupo
tau_global = np.sum(df_summary['n']*df_summary['tau_grupo'])
print("Efecto causal global:", float(tau_global))

Efectos causales por grupo:
 x
0    1.863225
1    2.043695
Name: tau_grupo, dtype: float64
Efecto causal global: 1.9570693860119674


## Mecanismo de asignación de tratamiento

<br>

Consideremos una variable indicadora de tratamiento para la unidad $i$, $Z_i$, la cuál es vectorizada como $Z = (Z_1, Z_2, ..., Z_n)$.

<br>

El resultado observado de la unidad $i$, $Y_i$, es una función de los resultados potenciales y del indicador de tratamiento $Z_i$.

$$\begin{align}
Y_i &= \begin{cases} Y_i(1), & \text{if } Z_i = 1 \\ Y_i(0), & \text{if } Z_i = 0 \end{cases}\\
&= Z_i Y_i(1) + (1 - Z_i)Y_i(0)\\
&= Y_i(0) + Z_i\{Y_i(1) - Y_i(0)\}\\
&= Y_i(0) + Z_i\tau_i.
\end{align}$$

En la última ecuación, puede observarse que los efectos causales pueden ser heterogéneos entre unidades.

En un experimento, al aplicarse el tratamiento binario, uno de los resultados potenciales se convierte en un dato faltante. Este dato faltante es el contrafactual (es decir, lo que no ocurre).

<br>

<span style="color:blue">**Definición: mecanismo de asignacion de tratamiento**</span>

El mecanismo de asignación de tratamiento es la distribución de probabilidad de $Z_i$

In [ ]:
# Ejemplo: Efectos potenciales y mecanismo de asignación de tratamiento

# Queremos mostrar el rol crucial del mecanismo de asignación de tratamiento
# sobre los estimadores de los efectos causales promedio

# Generamos los resultados potenciales y los efectos causales
n = 50
y_0 = np.random.normal(size=n)
tau = -0.5 + y_0
y_1 = y_0 + tau

# Doctor perfecto: asigna el tratamiento cuando el efecto causal es no negativo
z = np.where(tau >= 0, 1, 0) # Mecanismo de asignación, asigna cuando tau es mayor a 1
y = y_0 + z*tau # resultado observado

print("Diferencia de medias en el caso del doctor perfecto:",
      float(np.mean(y[z==1]) - np.mean(y[z==0])))

# Doctor ingenuo: asigna el tratamiento aleatoriamente
z = np.random.binomial(1, 0.5, n)
y = y_0 + z*tau # resultado observado

print("Diferencia de medias en el caso del doctor ingenuo:",
      float(np.mean(y[z==1]) - np.mean(y[z==0])))

Diferencia de medias en el caso del doctor perfecto: 2.409152425687915
Diferencia de medias en el caso del doctor ingenuo: -0.4969622684915165
